# Notebook 01: Environment Diagnostics and Dataset Discovery
## Hallucination-Resistant Medical Report Generation Using Generative AI

### Research Objectives:
1. Inspect Kaggle GPU capabilities (NVIDIA T4, P100, etc.), CUDA memory, and system RAM.
2. Safely verify required and optional dependencies with graceful fallbacks.
3. Discover mounted chest X-ray datasets in `/kaggle/input` (IU X-ray, MIMIC-CXR, CheXpert) or local fallback.
4. Perform strict patient-level splitting to prevent data leakage across train/val/test partitions.


In [ ]:
# 1. Module Path Setup and Environment Diagnostics
import sys
import os
from pathlib import Path

# Dynamically add project root to sys.path
for p in [Path.cwd(), Path.cwd().parent, Path('/kaggle/working/hallucination_medical_ai')]:
    if (p / 'src').exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))
        print(f'Configured sys.path with project root: {p}')

import torch
from src.environment import get_environment
from src.config import get_config

env = get_environment()
env.print_summary()
cfg = get_config()


### 2. Dependency Verification
Verify that key scientific packages (`torch`, `torchvision`, `transformers`, `sentence_transformers`, `faiss`) are available, and install lightweight Kaggle packages if missing.


In [ ]:
# Test optional Kaggle packages
missing_pkgs = []
for pkg in ['torchxrayvision', 'faiss', 'pydicom', 'rouge_score', 'bert_score']:
    try:
        __import__(pkg)
        print(f'[OK] {pkg} is installed')
    except ImportError:
        print(f'[MISSING] {pkg} not found')
        missing_pkgs.append(pkg)

if missing_pkgs:
    print(f'Install missing packages via: pip install {' '.join(missing_pkgs)}')


### 3. Dataset Discovery
Scan `/kaggle/input` or fallback `./data` to detect chest X-ray images, CSV metadata, and map them to standardized fields:
- `image_id`
- `image_path`
- `patient_id`
- `findings`
- `impression`
- `labels`
- `image_view`


In [ ]:
from src.datasets.discovery import DatasetDiscovery

discovery = DatasetDiscovery(cfg)
datasets_found = discovery.scan_inputs()
print('Datasets detected:', list(datasets_found.keys()))
discovery.display_summary()


### 4. Build Standardized Patient-Level Split Index
Prevent patient leakage by partitioning unique patient IDs into Train (70%), Val (10%), Test (20%).


In [ ]:
index_df = discovery.build_unified_index(max_samples=None)
print(f'Total indexed images: {len(index_df)}')
print(index_df.head(5))

# Save standardized index to persistent outputs
output_index_path = cfg.output_dir / 'metrics' / 'dataset_index.csv'
index_df.to_csv(output_index_path, index=False)
print(f'Standardized dataset index successfully exported to: {output_index_path}')
